In [0]:
import datatime as _dt

try:
    arrival_date = dbutils.widget.get("arrival_date")
except Exception:
    arrival_date = _dt.datetime.today().strftime("%Y-%m-%d")

try: 
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

try:
    base_volume = dbutils.widgets.get("base_volume")
except Exception as e:
    base_volume = f"/Volumes/{catalog}/{schema}/data"

from pyspark.sql.functions import lit, current_timestamp, to_date
from pyspark.sql import functions as F
import time

booking_path = f"{base_volume}/bookings_data/bookings_{arrival_date}.csv"
customer_path = f"{base_volume}/customer_data/customers_{arrival_date}.csv"
missing = []

try: 
    dbutils.fs.ls(booking_path)
except Exception:
    missing.append("booking_path")
try:
    dbutils.fs.ls(customer_path)
except Exception:
    missing.append("customer_path")
    
if len(missing) > 0:
    raise FileNotFoundError(f"Missing input files: {missing}")

# Maintain workflow execution metadata in Ops schema

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog}.ops.run_log (
    run_id STRING,
    arrival_date TIMESTAMP,
    stage STRING,
    status STRING,
    message STRING,
    recorded_at timestamp
    ) USING DELTA
""")
run_id = f"nb-validate-{arrival_date}-{int(time.time())}"
log_df = spark.createDataFrame([
    (run_id, arrival_date, "validate_inputs", "STARTED", "Inputs Validated")
], ["run_id", "arrival_date", "stage", "status", "message"])
log_df = log_df.withColumn("arrival_date", F.to_date)).withColumn("recorded_at", current_timestamp())
log_df.write.mode("append").saveAsTable(f"{catalog}.ops.run_log")    

print("Validation Successful:", booking_path, customer_path)
